# Worker pool from scratch

We build the pattern by hand to feel the queue + semaphore + sentinel mechanics.

In [ ]:
import asyncio, random, time

async def call(x):
    await asyncio.sleep(random.uniform(0.05, 0.2))
    return x*2

async def worker(q, out):
    while True:
        x = await q.get()
        if x is None:
            return
        out.append(await call(x))

async def main(n=20, k=5):
    q = asyncio.Queue()
    out = []
    workers = [asyncio.create_task(worker(q, out)) for _ in range(k)]
    for i in range(n): await q.put(i)
    for _ in range(k): await q.put(None)
    await asyncio.gather(*workers)
    return out

t = time.perf_counter()
res = await main(20, 5)
print(int((time.perf_counter()-t)*1000), 'ms', res[:5], '...')